# Nintendo Power Corpus Generation:

### Define hocr parser

In [2]:
from pathlib import Path
from bs4 import BeautifulSoup
import re

DOCUMENTS_DIR = Path("./documents")

def parse_hocr_file(file_path):
    '''
    Takes an hocr scan of a Nintendo Power issue and extracts page text and metadata
    '''
    html = file_path.read_text(encoding="utf-8", errors="replace")
    soup = BeautifulSoup(html, "html.parser")
    issue_match = re.search(
        r"Issue\s+(\d+)",
        file_path.name,
        re.IGNORECASE,
    )
    issue_number = int(issue_match.group(1)) if issue_match else None

    pages = []

    for page in soup.select(".ocr_page"):
        page_id = page.get("id", "")
        page_match = re.search(r"page_(\d+)", page_id)

        if not page_match:
            continue

        page_number = int(page_match.group(1))
        lines = page.select(".ocr_line, .ocr_caption")
        page_text = "\n".join(
            text
            for line in lines
            if (text := line.get_text(" ", strip=True))
        )

        pages.append({
            "filename": file_path.name,
            "issue_number": issue_number,
            "page_number": page_number,
            "text": page_text,
        })

    return sorted(pages, key=lambda page: page["page_number"])

### Define page grouping

In [4]:

def create_page_chunks(pages):
    '''
    Grab Left+Right page layouts. Most issues seem to have a coherent theme across each two page layout.
    '''
    pages_per_chunk = 2
    content_pages = sorted((page for page in pages if page["page_number"] > 0), key=lambda page: page["page_number"])
    page_numbers = [page["page_number"] for page in content_pages]
    if len(page_numbers) != len(set(page_numbers)):
        raise ValueError("Duplicate page numbers found in this issue.")

    if any(current != previous + 1 for previous, current in zip(page_numbers, page_numbers[1:])):
        raise ValueError("Missing page numbers found; inspect this issue before pairing.")

    chunks = []

    for start in range(0, len(content_pages) - 1, 2):
        page_group = content_pages[start:start + 2]

        chunk_text = "\n\n".join(f"[Scan page {page['page_number']}]\n{page['text']}" for page in page_group)

        if not any(page["text"].strip() for page in page_group):
            continue

        chunks.append({
            "text": chunk_text,
            "filename": page_group[0]["filename"],
            "issue_number": page_group[0]["issue_number"],
            "start_page": page_group[0]["page_number"],
            "end_page": page_group[-1]["page_number"],
            "pages": ",".join(
                str(page["page_number"]) for page in page_group
            ),
        })

    return chunks

##### Test

In [5]:

test_file = sorted(DOCUMENTS_DIR.glob("*.html"))[0]

test_pages = parse_hocr_file(test_file)
test_chunks = create_page_chunks(test_pages)

print(f"Issue file: {test_file.name}")
print(f"Pages extracted: {len(test_pages)}")
print(f"Two-page spreads created: {len(test_chunks)}")

for chunk in test_chunks[:3]:
    print(
        f"\nPages {chunk['pages']} "
        f"| Words: {len(chunk['text'].split())}"
    )
    print(chunk["text"][:500])
    print("-" * 60)


Issue file: Nintendo Power Issue 001_hocr.html
Pages extracted: 113
Two-page spreads created: 55

Pages 1,2 | Words: 376
[Scan page 1]
Powerline to the
206- А.
Мете always coming up with ways to help you play
a winning game. Our “powerline” to the Nintendo
game counselors is one of our best ideas yet. Іп fact,
we have so many video game fans calling that we're
putting more game experts on the team. So be
patient while we gear up to take more calls and
answer more of your questions than ever.
Get tips from the pros by
calling 206-885-7529
Г (that's 885-РІ ДҮ)
А Monday through Saturday:
0 «Тат to 11pm Eastern time
«
------------------------------------------------------------

Pages 3,4 | Words: 465
[Scan page 3]
lume; |жер |
FEATURES
SUPER MARIO BROS. 2 6
Be the first to get the inside info on Mario’s latest adventure.
SECOND re OF ZELDA — 26
Have you mastered the second quete
R.B.I. BASEBALL
(“5 BASES LOADED
MAJOR LEAGUE BASEBALL
COUNSELORS CORNER: 50
55
FREE CLASSIFIED ) INFORMATION —

In [7]:
content_pages = [page for page in test_pages if page["page_number"] > 0]

print("Pages including cover:", len(test_pages))
print("Content pages:", len(content_pages))
print("Pages with no OCR text:", sum(not page["text"].strip() for page in content_pages))
print("First page number:", min(p["page_number"] for p in content_pages))
print("Last page number:", max(p["page_number"] for p in content_pages))
print("Content page numbers:", [p["page_number"] for p in content_pages])

Pages including cover: 113
Content pages: 112
Pages with no OCR text: 5
First page number: 1
Last page number: 112
Content page numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112]


### Keyword tagging

In [11]:
import re

keywords = [
    "tip",
    "hint",
    "secret",
    "cheat",
    "trick",
    "shortcut",
    "password",
    "hidden",
    "walkthrough",
    "code",
    "game genie",
    "warp zone",
    "easter egg",
    "pak watch",
    "counselors",
    "classified information"
]

def find_strong_signals(text):
    '''
    Return identified keywords in ocr text.
    '''

    text_lower = text.lower()
    matches = []

    for keyword in keywords:
        pattern = r"\b" + r"\s+".join(re.escape(word) for word in keyword.split()) + r"s?\b"

        if re.search(pattern, text_lower):
            matches.append(keyword)

    return matches

##### Test

In [12]:
for chunk in test_chunks[:10]:
    matches = find_strong_signals(chunk["text"])

    print(
        f"Pages {chunk['pages']} | "
        f"Matches: {matches if matches else 'None'}"
    )

Pages 1,2 | Matches: ['tip', 'trick', 'counselors', 'classified information']
Pages 3,4 | Matches: ['pak watch', 'counselors']
Pages 5,6 | Matches: None
Pages 7,8 | Matches: ['trick']
Pages 9,10 | Matches: ['trick']
Pages 11,12 | Matches: None
Pages 13,14 | Matches: ['secret', 'shortcut']
Pages 15,16 | Matches: ['hint', 'secret', 'hidden']
Pages 17,18 | Matches: ['hint', 'secret']
Pages 19,20 | Matches: ['hint', 'secret', 'trick']


### Split issues, get keywords

In [14]:
from pathlib import Path
from tqdm.auto import tqdm

DOCUMENTS_DIR = Path("./documents")

hocr_files = sorted(DOCUMENTS_DIR.glob("*hocr.html"))

all_pages = []
all_chunks = []
file_errors = []

for file_path in tqdm(hocr_files, desc="Processing HOCR files", unit="file"):
    try:
        pages = parse_hocr_file(file_path)
        chunks = create_page_chunks(pages)

        all_pages.extend(pages)

        for chunk in chunks:
            matches = find_strong_signals(chunk["text"])

            chunk["keywords"] = matches
            chunk["keyword_match_count"] = len(matches)

        all_chunks.extend(chunks)

    except Exception as e:
        file_errors.append({
            "filename": file_path.name,
            "error": str(e),
        })

print(f"HOCR files found: {len(hocr_files)}")
print(f"Pages extracted: {len(all_pages)}")
print(f"Spreads created: {len(all_chunks)}")
print(f"Files with errors: {len(file_errors)}")
print(f"Spreads with keyword matches: {sum(bool(c['keywords']) for c in all_chunks)}")

Processing HOCR files:   0%|          | 0/285 [00:00<?, ?file/s]

HOCR files found: 285
Pages extracted: 35214
Spreads created: 17303
Files with errors: 0
Spreads with keyword matches: 8104


##### Check stats:

In [15]:
import statistics
from collections import Counter

word_counts = [len(chunk["text"].split()) for chunk in all_chunks]
empty_spreads = [chunk for chunk in all_chunks if not chunk["text"].strip()]
keyword_counts = Counter(keyword for chunk in all_chunks for keyword in chunk["keywords"])
spreads_with_keywords = sum(bool(chunk["keywords"])for chunk in all_chunks)

print("=== CORPUS SUMMARY ===")
print(f"HOCR files found:          {len(hocr_files)}")
print(f"Files with errors:         {len(file_errors)}")
print(f"Pages extracted:           {len(all_pages)}")
print(f"Spreads created:           {len(all_chunks)}")
print(f"Empty spreads:             {len(empty_spreads)}")
print(f"Spreads with keywords:     {spreads_with_keywords}")
print(f"Spreads without keywords:  {len(all_chunks) - spreads_with_keywords}")

if word_counts:
    print("\n=== WORD COUNT STATISTICS ===")
    print(f"Total words:  {sum(word_counts):,}")
    print(f"Minimum:      {min(word_counts):,}")
    print(f"Median:       {statistics.median(word_counts):,.1f}")
    print(f"Mean:         {statistics.mean(word_counts):,.1f}")
    print(f"Maximum:      {max(word_counts):,}")

print("\n=== KEYWORD FREQUENCIES ===")
for keyword, count in keyword_counts.most_common():
    print(f"{keyword:<25} {count:>6,}")

if file_errors:
    print("\n=== FILE ERRORS ===")
    for error in file_errors:
        print(f"{error['filename']}: {error['error']}")
else:
    print("\nNo file-processing errors were recorded.")

=== CORPUS SUMMARY ===
HOCR files found:          285
Files with errors:         0
Pages extracted:           35214
Spreads created:           17303
Empty spreads:             0
Spreads with keywords:     8104
Spreads without keywords:  9199

=== WORD COUNT STATISTICS ===
Total words:  11,039,405
Minimum:      7
Median:       601.0
Mean:         638.0
Maximum:      3,550

=== KEYWORD FREQUENCIES ===
secret                     3,101
hidden                     2,277
trick                      1,778
tip                        1,661
code                       1,450
password                     815
hint                         446
classified information       433
shortcut                     421
counselors                   392
pak watch                    351
cheat                        349
warp zone                     60
easter egg                    25
walkthrough                   20
game genie                     2

No file-processing errors were recorded.


### Save so I don't have to reprocess:

In [16]:
import json
from pathlib import Path

OUTPUT_DIR = Path("./processed_documents")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

corpus_path = OUTPUT_DIR / "nintendo_power_spreads.jsonl"

with corpus_path.open("w", encoding="utf-8") as f:
    for chunk in all_chunks:
        record = {
            "id": (
                f"issue_{chunk['issue_number']}_"
                f"pages_{chunk['start_page']}_{chunk['end_page']}"
            ),
            "text": chunk["text"],
            "filename": chunk["filename"],
            "issue_number": chunk["issue_number"],
            "start_page": chunk["start_page"],
            "end_page": chunk["end_page"],
            "pages": chunk["pages"],
            "keywords": chunk["keywords"],
            "keyword_match_count": chunk["keyword_match_count"],
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(all_chunks):,} spreads to {corpus_path}")
print(f"File size: {corpus_path.stat().st_size / (1024**2):.2f} MB")

Saved 17,303 spreads to processed_documents\nintendo_power_spreads.jsonl
File size: 65.02 MB


In [17]:
with corpus_path.open("r", encoding="utf-8") as f:
    first_record = json.loads(next(f))

print("Record ID:", first_record["id"])
print("Source:", first_record["filename"])
print("Pages:", first_record["pages"])
print("Keywords:", first_record["keywords"])
print("Text preview:")
print(first_record["text"][:500])

Record ID: issue_1_pages_1_2
Source: Nintendo Power Issue 001_hocr.html
Pages: 1,2
Keywords: ['tip', 'trick', 'counselors', 'classified information']
Text preview:
[Scan page 1]
Powerline to the
206- А.
Мете always coming up with ways to help you play
a winning game. Our “powerline” to the Nintendo
game counselors is one of our best ideas yet. Іп fact,
we have so many video game fans calling that we're
putting more game experts on the team. So be
patient while we gear up to take more calls and
answer more of your questions than ever.
Get tips from the pros by
calling 206-885-7529
Г (that's 885-РІ ДҮ)
А Monday through Saturday:
0 «Тат to 11pm Eastern time
«


### Load from file

In [1]:
import json
from pathlib import Path

# Paths
INPUT_PATH = Path("./processed_documents/nintendo_power_spreads.jsonl")
FILTERED_PATH = Path("./processed_documents/nintendo_power_tagged_spreads.jsonl")

# Load all spreads
with INPUT_PATH.open("r", encoding="utf-8") as f:
    all_records = [json.loads(line) for line in f if line.strip()]

# Keep oonly tagged spreads
tagged_records = [record for record in all_records if record.get("keyword_match_count", 0) > 0]

with FILTERED_PATH.open("w", encoding="utf-8") as f:
    for record in tagged_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Original spreads: {len(all_records):,}")
print(f"Tagged spreads:   {len(tagged_records):,}")
print(f"Excluded spreads: {len(all_records) - len(tagged_records):,}")
print(f"Filtered corpus saved to: {FILTERED_PATH}")
print(f"Filtered file size: {FILTERED_PATH.stat().st_size / (1024**2):.2f} MB")


Original spreads: 17,303
Tagged spreads:   8,104
Excluded spreads: 9,199
Filtered corpus saved to: processed_documents\nintendo_power_tagged_spreads.jsonl
Filtered file size: 35.92 MB


### Generate Langchain docs:

In [2]:
from langchain_core.documents import Document

documents = []

for record in tagged_records:
    metadata = {
        "id": record["id"],
        "filename": record["filename"],
        "issue_number": record["issue_number"],
        "start_page": record["start_page"],
        "end_page": record["end_page"],
        "pages": record["pages"],
        "keywords": ", ".join(record["keywords"]),
        "keyword_match_count": record["keyword_match_count"],
    }

    documents.append(
        Document(
            page_content=record["text"],
            metadata=metadata,
        )
    )

print(f"LangChain documents created: {len(documents):,}")

print("\n=== SAMPLE DOCUMENT ===")
print("ID:", documents[0].metadata["id"])
print("Source:", documents[0].metadata["filename"])
print("Pages:", documents[0].metadata["pages"])
print("Keywords:", documents[0].metadata["keywords"])
print("\nText preview:")
print(documents[0].page_content[:500])


LangChain documents created: 8,104

=== SAMPLE DOCUMENT ===
ID: issue_1_pages_1_2
Source: Nintendo Power Issue 001_hocr.html
Pages: 1,2
Keywords: tip, trick, counselors, classified information

Text preview:
[Scan page 1]
Powerline to the
206- А.
Мете always coming up with ways to help you play
a winning game. Our “powerline” to the Nintendo
game counselors is one of our best ideas yet. Іп fact,
we have so many video game fans calling that we're
putting more game experts on the team. So be
patient while we gear up to take more calls and
answer more of your questions than ever.
Get tips from the pros by
calling 206-885-7529
Г (that's 885-РІ ДҮ)
А Monday through Saturday:
0 «Тат to 11pm Eastern time
«


### Generate Embeddings:

In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embeddings = OpenAIEmbeddings(
    model=os.getenv("DEFAULT_EMBEDDINGS_MODEL"),
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
)

test_vector = embeddings.embed_query(
    documents[0].page_content
)

print("Embedding test successful!")
print("Vector dimensions:", len(test_vector))
print("First 5 values:", test_vector[:5])

Embedding test successful!
Vector dimensions: 3072
First 5 values: [0.0145111083984375, -0.06695556640625, -0.0198516845703125, -0.01047515869140625, -0.05255126953125]


In [7]:
from langchain_chroma import Chroma

# small test collection
test_vectorstore = Chroma(
    collection_name="nintendo_power_test",
    embedding_function=embeddings,
    persist_directory="./chroma_db_nintendo_power",
)

test_documents = documents[:10]

ids = [document.metadata["id"] for document in test_documents]

test_vectorstore.add_documents(
    documents=test_documents,
    ids=ids,
)

print(f"Documents added: {len(test_documents)}")
print(f"Collection count: {test_vectorstore._collection.count()}")

Documents added: 10
Collection count: 10


In [8]:
query = "Where can I find tips, secrets, and tricks for Nintendo games?"

results = test_vectorstore.similarity_search(
    query,
    k=3,
)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("ID:", doc.metadata["id"])
    print("Issue:", doc.metadata["issue_number"])
    print("Pages:", doc.metadata["pages"])
    print("Keywords:", doc.metadata["keywords"])
    print("Text:", doc.page_content[:400])


--- Result 1 ---
ID: issue_1_pages_13_14
Issue: 1
Pages: 13,14
Keywords: secret, shortcut
Text: [Scan page 13]
Now we'll give you some important pointers оп how to
tackle each area in Super Mario Bros. 2. To be on the safe
side, try out each character, and get to know their
movements at the very beginning of the game. Once you
pass through the door below, the real fast-action fun
starts. The first area will be an easy win.
V IMPROVE YOUR JUMPING SKILLS
You'll find two places where your fall 

--- Result 2 ---
ID: issue_1_pages_15_16
Issue: 1
Pages: 15,16
Keywords: hint, secret, hidden
Text: [Scan page 15]
Right after you start your adventure in this area, you will
find yourself in the sky. Consider your options. There is
only one smart way to go. You must take the flying carpet
away from Pidgit. Later, when the going gets tough, try
using bombs. They will make a big difference.
SECRET ROOM)
! SECRET HINT)
1 Aim and throw the Magic Potion between the two
jars. Once you enter into "Sub


##### Embed the full corpus:

In [9]:
from langchain_chroma import Chroma
from tqdm.auto import tqdm

BATCH_SIZE = 100

vectorstore = Chroma(
    collection_name="nintendo_power_secrets",
    embedding_function=embeddings,
    persist_directory="./chroma_db_nintendo_power",
)

for start in tqdm(
    range(0, len(documents), BATCH_SIZE),
    desc="Embedding Nintendo Power spreads",
):
    batch = documents[start : start + BATCH_SIZE]
    batch_ids = [doc.metadata["id"] for doc in batch]

    vectorstore.add_documents(
        documents=batch,
        ids=batch_ids,
    )

print(f"Documents in collection: {vectorstore._collection.count():,}")

Embedding Nintendo Power spreads:   0%|          | 0/82 [00:00<?, ?it/s]

Documents in collection: 8,104


##### Check full corpus:

In [11]:
print(f"Collection count: {vectorstore._collection.count()}")

Collection count: 8104


In [10]:
query = "How can I find secret rooms and hidden areas in Super Mario Bros. 2?"

results = vectorstore.similarity_search(
    query,
    k=5,
)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("ID:", doc.metadata["id"])
    print("Issue:", doc.metadata["issue_number"])
    print("Pages:", doc.metadata["pages"])
    print("Keywords:", doc.metadata["keywords"])
    print("Text:", doc.page_content[:500])


--- Result 1 ---
ID: issue_44_pages_147_148
Issue: 44
Pages: 147,148
Keywords: secret, hidden
Text: [Scan page 147]
Solve more Mario Land
mysteries. There
doors to secret stages in
two Pumpkin Zone
courses. You'll find them
by exploring out-of-the-
way areas and using spe-
cial Mario moves. Once
this pair of secret stages
has been cleared, the
markers on the Pumpkin
Zone map will produce
menacing spooks.
are
ledge as
You'll encounter
this pool shortly
after you ring the
midpoint Bell in the sec-
ond Pumpkin Zone stage.
Sink into the water and
seek out a secret stage
Avoid the spooks as you
cro

--- Result 2 ---
ID: issue_13_pages_21_22
Issue: 13
Pages: 21,22
Keywords: secret, hidden
Text: [Scan page 21]
Climb A Vine To A Secret Room
Just like in the original Super Mario Bros., some blocks have vines
inside of them. The vine in this world leads to a path of clouds that
will take Mario to a sky pipe and to a bonus coin room.
Talk about getting no respect! Because
of their chain, these C

### Generate Full Corpus (not just tagged spreads)

In [1]:
import json
from pathlib import Path
from langchain_core.documents import Document

INPUT_PATH = Path("./processed_documents/nintendo_power_spreads.jsonl")

with INPUT_PATH.open("r", encoding="utf-8") as f:
    all_records = [json.loads(line) for line in f if line.strip()]

documents = []

for record in all_records:
    metadata = {
        "id": record["id"],
        "filename": record["filename"],
        "issue_number": record["issue_number"],
        "start_page": record["start_page"],
        "end_page": record["end_page"],
        "pages": record["pages"],
        "keywords": ", ".join(record["keywords"]),
        "keyword_match_count": record["keyword_match_count"],
        "is_tagged": record["keyword_match_count"] > 0,
    }

    documents.append(
        Document(
            page_content=record["text"],
            metadata=metadata,
        )
    )

tagged_count = sum(doc.metadata["is_tagged"] for doc in documents)

print(f"Total documents: {len(documents):,}")
print(f"Tagged spreads: {tagged_count:,}")
print(f"Untagged spreads: {len(documents) - tagged_count:,}")

Total documents: 17,303
Tagged spreads: 8,104
Untagged spreads: 9,199


In [3]:
from langchain_chroma import Chroma
from tqdm.auto import tqdm
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embeddings = OpenAIEmbeddings(
    model=os.getenv("DEFAULT_EMBEDDINGS_MODEL"),
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
)

COLLECTION_NAME = "nintendo_power_all_spreads"
PERSIST_DIRECTORY = "./chroma_db_nintendo_power"
BATCH_SIZE = 100

vectorstore_all = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=PERSIST_DIRECTORY,
)

for start in tqdm(
    range(0, len(documents), BATCH_SIZE),
    desc="Embedding all Nintendo Power spreads",
):
    batch = documents[start : start + BATCH_SIZE]
    batch_ids = [doc.metadata["id"] for doc in batch]

    vectorstore_all.add_documents(
        documents=batch,
        ids=batch_ids,
    )

print(f"Collection: {COLLECTION_NAME}")
print(f"Documents in collection: {vectorstore_all._collection.count():}")

Embedding all Nintendo Power spreads:   0%|          | 0/174 [00:00<?, ?it/s]

Collection: nintendo_power_all_spreads
Documents in collection: 17303


In [5]:
print(f"Collection count: {vectorstore_all._collection.count()}")

Collection count: 17303


### Read in Nintendo Power HOCR files and parse to grab text:

In [1]:
from pathlib import Path
from bs4 import BeautifulSoup
import re

DOCUMENTS_DIR = Path("./documents")

def parse_hocr_file(file_path):
    """Extract text page by page from a HOCR HTML file."""

    html = file_path.read_text(encoding="utf-8", errors="replace")
    soup = BeautifulSoup(html, "html.parser")

    # Extract the issue number from filename
    issue_match = re.search(r"Issue\s+(\d+)", file_path.name, re.IGNORECASE)
    issue_number = int(issue_match.group(1)) if issue_match else None

    pages = []

    for page in soup.select(".ocr_page"):
        page_id = page.get("id", "")
        page_match = re.search(r"page_(\d+)", page_id)
        if not page_match:
            continue

        page_number = int(page_match.group(1))

        # Extract OCR lines in their original HTML order.
        lines = page.select(".ocr_line, .ocr_caption")

        if lines:
            page_text = "\n".join(
                line.get_text(" ", strip=True)
                for line in lines
                if line.get_text(" ", strip=True)
            )
        else:
            page_text = page.get_text(" ", strip=True)

        pages.append({
            "filename": file_path.name,
            "issue_number": issue_number,
            "page_number": page_number,
            "text": page_text,
        })

    return pages

In [5]:
def create_page_chunks(pages, pages_per_chunk=3, overlap=1):
    """Combine consecutive scan pages into overlapping text chunks."""
    if overlap >= pages_per_chunk:
        raise ValueError("Overlap must be smaller than chunk size.")

    chunks = []
    step = pages_per_chunk - overlap

    for start in range(0, len(pages), step):
        page_group = pages[start:start + pages_per_chunk]

        if not page_group:
            continue

        # Keep page boundaries visible in the text sent to the embedding model.
        chunk_text = "\n\n".join(
            f"[Scan page {page['page_number']}]\n{page['text']}"
            for page in page_group
        )

        if not any(page["text"].strip() for page in page_group):
            continue

        chunks.append({
            "text": chunk_text,
            "filename": page_group[0]["filename"],
            "issue_number": page_group[0]["issue_number"],
            "start_page": page_group[0]["page_number"],
            "end_page": page_group[-1]["page_number"],
            "pages": ",".join(
                str(page["page_number"]) for page in page_group
            ),
        })

        if start + pages_per_chunk >= len(pages):
            break

    return chunks

In [15]:
from pathlib import Path

DOCUMENTS_DIR = Path("./documents")

all_pages = []
all_chunks = []

hocr_files = sorted(DOCUMENTS_DIR.glob("*.html"))

print(f"Found {len(hocr_files)} HTML files")

for file_path in hocr_files:
    file_pages = parse_hocr_file(file_path)
    file_chunks = create_page_chunks(file_pages)

    all_pages.extend(file_pages)
    all_chunks.extend(file_chunks)

    print(
        f"{file_path.name}: "
        f"{len(file_pages)} pages, "
        f"{len(file_chunks)} chunks"
    )

print(f"\nTotal pages: {len(all_pages)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"Unique source files: {len(set(c['filename'] for c in all_chunks))}")

Found 5 HTML files
Nintendo Power Issue 148 (September 2001)_hocr.html: 137 pages, 68 chunks
Nintendo Power Issue 160 (September 2002)_hocr.html: 189 pages, 94 chunks
Nintendo Power Issue 173 (November 2003)_hocr.html: 175 pages, 87 chunks
Nintendo Power Issue 265 (March 2011)_hocr.html: 103 pages, 51 chunks
Nintendo Power Issue 48 (May 1993)_hocr.html: 123 pages, 61 chunks

Total pages: 727
Total chunks: 361
Unique source files: 5


In [17]:
from langchain_core.documents import Document

documents = []
document_ids = []

for chunk in all_chunks:
    documents.append(
        Document(
            page_content=chunk["text"],
            metadata={
                "filename": chunk["filename"],
                "issue_number": chunk["issue_number"],
                "start_page": chunk["start_page"],
                "end_page": chunk["end_page"],
                "pages": chunk["pages"],
            },
        )
    )

    document_ids.append(
        f"issue_{chunk['issue_number']}_page_{chunk['start_page']}"
    )

print(f"Documents prepared: {len(documents)}")
print(f"Unique IDs: {len(set(document_ids))}")

Documents prepared: 361
Unique IDs: 361


In [11]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

UNC_AI_BASE_URL = os.getenv("LLM_BASE_URL")
UNC_AI_API_KEY = os.getenv("LLM_API_KEY")

assert UNC_AI_API_KEY, "Missing UNC_AI_API_KEY — check your .env file"

EMBED_MODEL = os.getenv("DEFAULT_EMBEDDINGS_MODEL")

embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=UNC_AI_BASE_URL,
    api_key=UNC_AI_API_KEY,
)

print(f"Embedding model configured: {EMBED_MODEL}")


Embedding model configured: text-embedding-3-large


### Create Embeddings

In [19]:

from langchain_chroma import Chroma

COLLECTION_NAME = "nintendo_power_large"
PERSIST_DIRECTORY = "./chromadb"
BATCH_SIZE = 20

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIRECTORY,
    embedding_function=embeddings,
)

for start in range(0, len(documents), BATCH_SIZE):
    batch_documents = documents[start:start + BATCH_SIZE]
    batch_ids = document_ids[start:start + BATCH_SIZE]

    vectorstore.add_documents(
        documents=batch_documents,
        ids=batch_ids,
    )

    print(
        f"Embedded documents "
        f"{start + 1}-{start + len(batch_documents)} "
        f"of {len(documents)}"
    )

print(f"\nCollection: {COLLECTION_NAME}")
print(f"Stored embeddings: {vectorstore._collection.count()}")


Embedded documents 1-20 of 361
Embedded documents 21-40 of 361
Embedded documents 41-60 of 361
Embedded documents 61-80 of 361
Embedded documents 81-100 of 361
Embedded documents 101-120 of 361
Embedded documents 121-140 of 361
Embedded documents 141-160 of 361
Embedded documents 161-180 of 361
Embedded documents 181-200 of 361
Embedded documents 201-220 of 361
Embedded documents 221-240 of 361
Embedded documents 241-260 of 361
Embedded documents 261-280 of 361
Embedded documents 281-300 of 361
Embedded documents 301-320 of 361
Embedded documents 321-340 of 361
Embedded documents 341-360 of 361
Embedded documents 361-361 of 361

Collection: nintendo_power_large
Stored embeddings: 361


In [20]:
loaded_vectorstore = Chroma(
    collection_name="nintendo_power_large",
    persist_directory="./chromadb",
    embedding_function=embeddings,
)

print(f"Reloaded embeddings: {loaded_vectorstore._collection.count()}")

Reloaded embeddings: 361
